# 🌾 KisaanSathi: MobileNetV2 Training Pipeline (Google Colab GPU)
**Crop Disease Classification & Grad-CAM Visual Explainability**  

This notebook trains an industrial-grade **MobileNetV2** transfer learning classifier using your pre-split PlantVillage dataset stored on Google Drive (`My Drive/KisaanSathi/data/processed`).

### Pipeline Structure:
1. **GPU Acceleration Check** (T4 GPU enabled)
2. **Google Drive Mount** (Direct connection to `KisaanSathi/data/processed`)
3. **Dataset Ingestion** (`train`, `val`, `test` splits)
4. **MobileNetV2 Model Architecture** (Data Augmentation + `[-1, 1]` Rescaling + Base Model + Dropout + Dense Softmax)
5. **Stage 1 Training**: Feature extraction with frozen base ($lr = 10^{-3}$)
6. **Stage 2 Training**: Fine-tuning top 30 layers ($lr = 10^{-5}$)
7. **Test Set Evaluation**: Accuracy, F1-scores, and Classification Report
8. **Artifact Export**: Direct save to Google Drive & local download for KisaanSathi

In [ ]:
# STEP 1: Verify GPU Acceleration
import os
os.system('nvidia-smi')
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU Devices:", gpus)
if not gpus:
    print("⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")
else:
    print("✅ GPU is ready for fast training!")

In [ ]:
# STEP 2: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Set base dataset directory matching your Drive upload
DATA_DIR = "/content/drive/MyDrive/KisaanSathi/data/processed"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

# Verify path and count classes
if os.path.exists(TRAIN_DIR):
    classes = sorted(os.listdir(TRAIN_DIR))
    print(f"✅ Successfully located dataset at: {DATA_DIR}")
    print(f"Detected {len(classes)} classes in train folder:")
    for i, c in enumerate(classes[:5]):
        print(f"   [{i}] {c}")
    print(f"   ... and {len(classes) - 5} more")
else:
    print(f"❌ ERROR: Path not found: {TRAIN_DIR}. Please check your Drive folder name!")

In [ ]:
# STEP 3: Load Datasets into tf.data Pipelines
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
AUTOTUNE = tf.data.AUTOTUNE

print("Loading Training set...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

print("Loading Validation set...")
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

print("Loading Test set...")
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\n✅ Total classes loaded: {num_classes}")

# Optimize pipeline with prefetching
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# Save class mapping JSON directly
import json
os.makedirs("/content/artifacts", exist_ok=True)
with open("/content/artifacts/class_names.json", "w", encoding="utf-8") as f:
    json.dump({str(i): name for i, name in enumerate(class_names)}, f, indent=2)
print("✅ Saved class mapping to /content/artifacts/class_names.json")

In [ ]:
# STEP 4: Build MobileNetV2 Architecture
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# 1. Real-time data augmentation for training robustness
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")

# 2. Pretrained MobileNetV2 Base (ImageNet)
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Frozen for Stage 1

# 3. Full Model Assembly
inputs = layers.Input(shape=(224, 224, 3), name="input_image")
x = data_augmentation(inputs)
# Rescaling to [-1, 1] required by MobileNetV2
x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_rescaling")(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dropout(0.3, name="dropout")(x)
outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

model = models.Model(inputs, outputs, name="KisaanSathi_MobileNetV2")
model.summary()

In [ ]:
# STEP 5: STAGE 1 Training (Feature Extraction - Frozen Base)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_stage1 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint("/content/artifacts/stage1_best.keras", monitor="val_accuracy", save_best_only=True, verbose=1)
]

print("Starting Stage 1 training (Base model frozen)...")
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks_stage1
)
print("✅ Stage 1 complete!")

In [ ]:
# STEP 6: STAGE 2 Training (Fine-Tuning Top 30 Layers)
base_model.trainable = True

# Freeze all layers except top 30, and keep BatchNormalization frozen
unfreeze_layers = 30
for layer in base_model.layers[:-unfreeze_layers]:
    layer.trainable = False
for layer in base_model.layers[-unfreeze_layers:]:
    if isinstance(layer, layers.BatchNormalization) or "BatchNormalization" in layer.__class__.__name__:
        layer.trainable = False
    else:
        layer.trainable = True

trainable_count = sum(len(l.trainable_weights) for l in model.layers)
print(f"Fine-tuning active with top {unfreeze_layers} layers unfreezed.")

# Recompile with lower learning rate for stable fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_stage2 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.ModelCheckpoint("/content/artifacts/crop_disease_model.keras", monitor="val_accuracy", save_best_only=True, verbose=1)
]

print("Starting Stage 2 fine-tuning...")
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks_stage2
)

# Save final model
model.save("/content/artifacts/crop_disease_model.keras")
print("✅ Stage 2 complete! Final model saved to /content/artifacts/crop_disease_model.keras")

In [ ]:
# STEP 7: Evaluate on Unseen Test Set
import numpy as np
from sklearn.metrics import classification_report

print("Evaluating model on unseen TEST set...")
test_loss, test_acc = model.evaluate(test_ds)
print(f"\n🎯 Test Accuracy: {test_acc * 100:.2f}%")
print(f"🎯 Test Loss: {test_loss:.4f}")

# Classification report
y_true, y_pred = [], []
for imgs, labels in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# STEP 8: Save Model Artifacts to Google Drive & Download Locally
import shutil
from google.colab import files

# 1. Save directly back to your Google Drive folder so you never lose it
drive_save_dir = "/content/drive/MyDrive/KisaanSathi/models"
os.makedirs(drive_save_dir, exist_ok=True)
shutil.copy("/content/artifacts/crop_disease_model.keras", os.path.join(drive_save_dir, "crop_disease_model.keras"))
shutil.copy("/content/artifacts/class_names.json", os.path.join(drive_save_dir, "class_names.json"))
print(f"✅ Backup saved to Google Drive: {drive_save_dir}")

# 2. Trigger direct browser download to your computer
print("Downloading artifacts to your PC...")
files.download("/content/artifacts/crop_disease_model.keras")
files.download("/content/artifacts/class_names.json")
print("\n🎉 Training Complete!")
print("Next Step: Copy crop_disease_model.keras into KisaanSathi/ml/models/ on your computer.")